In [27]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EcommerceAnalysis") \
    .getOrCreate()

In [28]:
spark.sql("DROP TABLE IF EXISTS customers")
spark.sql("DROP TABLE IF EXISTS products")
spark.sql("DROP TABLE IF EXISTS orders")

spark.sql("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INT,
    name STRING,
    region STRING,
    signup_date DATE
)
USING PARQUET
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS products (
    product_id INT,
    product_name STRING,
    category STRING,
    price DECIMAL(10,2)
)
USING PARQUET
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS orders (
    order_id INT,
    customer_id INT,
    order_date DATE,
    product_id INT,
    quantity INT
)
USING PARQUET
""")

DataFrame[]

In [29]:
import random
from datetime import datetime, timedelta

customers_data = []
products_data = []
orders_data = []

# customers
for i in range(1,1001):
    customers_data.append((
        i,
        f"Customer{i}",
        random.choice(['North','South','East','West']),
        f"2025-01-{random.randint(1,28)}"
    ))

# products
for i in range(1,201):
    products_data.append((
        i,
        f"Product{i}",
        random.choice(['Electronics','Clothing','Home']),
        random.randint(10,2000)
    ))

# orders
for i in range(1,5001):
    date = datetime(2026,1,1) + timedelta(days=random.randint(0,60))
    orders_data.append((
        i,
        random.randint(1,1000),
        str(date.date()),
        random.randint(1,200),
        random.randint(1,5)
    ))

In [30]:
customers_df = spark.createDataFrame(
    customers_data,
    ["customer_id","name","region","signup_date"]
)

products_df = spark.createDataFrame(
    products_data,
    ["product_id","product_name","category","price"]
)

orders_df = spark.createDataFrame(
    orders_data,
    ["order_id","customer_id","order_date","product_id","quantity"]
)

In [31]:
customers_df.write.mode("overwrite").saveAsTable("customers")

products_df.write.mode("overwrite").saveAsTable("products")

orders_df.write.mode("overwrite").saveAsTable("orders")

In [33]:
customers_df.createOrReplaceTempView("customers_df")
products_df.createOrReplaceTempView("products_df")
orders_df.createOrReplaceTempView("orders_df")

In [34]:
spark.sql("SELECT COUNT(*) FROM customers_df").show()
spark.sql("SELECT COUNT(*) FROM products_df").show()
spark.sql("SELECT COUNT(*) FROM orders_df").show()

+--------+
|count(1)|
+--------+
|    1000|
+--------+

+--------+
|count(1)|
+--------+
|     200|
+--------+

+--------+
|count(1)|
+--------+
|    5000|
+--------+



In [35]:
spark.sql("SHOW TABLES").show()

+---------+------------+-----------+
|namespace|   tableName|isTemporary|
+---------+------------+-----------+
|  default|   customers|      false|
|  default|      orders|      false|
|  default|    products|      false|
|         |customers_df|       true|
|         |   orders_df|       true|
|         | products_df|       true|
+---------+------------+-----------+



## Monthly revenue

In [36]:
spark.sql("""
Select 
date_format(order_date,'MMM') AS month,
sum(quantity*price) as total_revenue
From orders as o
join products as p on o.product_id = p.product_id
group by month
order by month;
""").show()


+-----+-------------+
|month|total_revenue|
+-----+-------------+
|  Feb|      6476260|
|  Jan|      7059658|
|  Mar|       479019|
+-----+-------------+



#  Top selling products

In [37]:
spark.sql("""
SELECT * from products
""").show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|         1|    Product1|Electronics|   85|
|         2|    Product2|Electronics| 1519|
|         3|    Product3|   Clothing|  801|
|         4|    Product4|Electronics| 1077|
|         5|    Product5|       Home|  753|
|         6|    Product6|Electronics| 1530|
|         7|    Product7|Electronics|  334|
|         8|    Product8|Electronics| 1246|
|         9|    Product9|       Home|  198|
|        10|   Product10|   Clothing|  447|
|        11|   Product11|Electronics| 1086|
|        12|   Product12|       Home|  607|
|        13|   Product13|       Home| 1719|
|        14|   Product14|Electronics|  171|
|        15|   Product15|   Clothing| 1131|
|        16|   Product16|   Clothing| 1207|
|        17|   Product17|Electronics| 1851|
|        18|   Product18|Electronics|  448|
|        19|   Product19|       Home|  535|
|        20|   Product20|   Clot

In [38]:
spark.sql("""
SELECT p.product_name AS product,
       SUM(o.quantity) AS quantity
FROM orders o
JOIN products p
ON o.product_id = p.product_id
GROUP BY p.product_name
ORDER BY quantity DESC
""").show()

+----------+--------+
|   product|quantity|
+----------+--------+
| Product15|     115|
|Product116|     110|
|Product165|     109|
| Product33|     107|
|Product118|     106|
|Product145|     105|
|Product103|     105|
|Product170|     102|
|Product107|     101|
|Product164|     101|
|  Product4|     101|
| Product21|     100|
| Product65|      99|
|Product108|      99|
| Product37|      98|
|Product159|      98|
| Product35|      98|
|Product139|      97|
| Product10|      97|
|Product133|      95|
+----------+--------+
only showing top 20 rows



# Revenue by region

In [39]:
spark.sql("""
select c.region AS region,
      sum(o.quantity * p.price) AS revenue
from customers c
join orders o
on c.customer_id = o.customer_id
join products p
on o.product_id = p.product_id
group by c.region
order by revenue DESC
""").show()

+------+-------+
|region|revenue|
+------+-------+
|  East|3698484|
|  West|3684012|
| South|3591558|
| North|3040883|
+------+-------+



# Repeat customers

In [42]:
spark.sql("""
SELECT c.name, c.customer_id,
       COUNT(o.order_id) AS total_orders
FROM orders o
JOIN customers c 
     ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.name
HAVING COUNT(o.order_id) > 1;
""").show()

+-----------+-----------+------------+
|       name|customer_id|total_orders|
+-----------+-----------+------------+
|Customer981|        981|           7|
|Customer972|        972|           5|
|Customer108|        108|           3|
|Customer561|        561|           8|
|Customer583|        583|           2|
|Customer854|        854|           4|
|Customer541|        541|           5|
|Customer319|        319|           4|
|Customer755|        755|           3|
|Customer962|        962|           4|
|Customer286|        286|           3|
|Customer578|        578|          10|
|Customer343|        343|           6|
|Customer845|        845|           7|
|Customer897|        897|           5|
|Customer138|        138|           9|
|Customer446|        446|           2|
|Customer134|        134|          10|
|Customer260|        260|          10|
|Customer650|        650|           6|
+-----------+-----------+------------+
only showing top 20 rows

